# Duration of response — depth is not durability

**NOT FOR CLINICAL USE.** Population/trial-level rates only; no individual response or duration, no therapy ranking. Executed in CI (nbmake).

v0.27 added ORR and showed it is a *conditional* OS surrogate. This notebook supplies the missing dimension and the *mechanism*: **duration of response (DoR)** — how long responses last. ORR measures response **breadth** (how many respond); DoR measures **durability** (how long). They dissociate, and the ORR-surrogate failures are durability failures: the model with the **highest ORR has the shortest DoR**, which is why it has the worst tail-driven survival. See `docs/specs/research/duration-of-response.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.response import response_episode, objective_response_rate

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 208, 417)
print(f'{len(ds)} records | onkos {onkos.__version__}')

## DoR, consistently defined with the response category

`response_episode(t, v)` returns the RECIST best response AND the duration of response from the same observed-baseline trajectory: DoR = time from PR onset (SLD ≤70% of baseline) to progression (SLD ≥120% of nadir). A response that never progresses is right-censored (nan), not zero — the durable responders DoR most wants to count.

In [ ]:
ts = np.linspace(0, 104, 209)
# a PR that onsets early and progresses later
v = np.concatenate([np.linspace(100, 50, 53), np.full(80, 50.0), np.linspace(50, 90, 76)])
print('PR -> PD:', response_episode(ts, v))
# a durable PR that never regrows -> censored (nan)
vd = np.concatenate([np.linspace(100, 40, 100), np.full(109, 40.0)])
print('durable PR (censored):', response_episode(ts, vd))

## Breadth (ORR) vs durability (DoR) dissociate

The highest-ORR model is *not* the most durable. The two-population (mechanistic resistance) model responds in everyone (ORR 1.0) but briefly (short DoR) — a deep early shrink then fast resistant regrowth.

In [ ]:
print(f"{'model':<22}{'ORR':>6}{'DoR':>6}{'cens':>6}{'OS(wk8)':>8}")
for rid in ['drug_effect.norton_simon.nsclc','resistance.claret_2009.tgi','resistance.nsclc_first_line.two_population','tgi_metrics.wang_2009.biexponential']:
    rr = objective_response_rate(ds, rid, context=ctx, t=t, n=400)
    dor = 'n/r' if rr.median_dor_weeks is None else f'{rr.median_dor_weeks:.0f}'
    print(f"{rid.split('.')[1]:<22}{rr.orr:>6.2f}{dor:>6}{rr.dor_censored_fraction:>6.0%}{rr.median_os_weeks:>8.0f}")

# Depth is not durability: highest ORR is not the most durable.
two = objective_response_rate(ds, 'resistance.nsclc_first_line.two_population', context=ctx, t=t, n=400)
clr = objective_response_rate(ds, 'resistance.claret_2009.tgi', context=ctx, t=t, n=400)
assert two.orr >= clr.orr and two.median_dor_weeks < clr.median_dor_weeks
print('\ndepth != durability: two-population ORR >= Claret but DoR shorter')

## DoR is the mechanism of the ORR → OS surrogate failure

Under the tail-sensitive k_g link (v0.27), ORR mis-ranks OS — the highest responder has the shortest OS. DoR explains it: that model's responses are the briefest, while the longest-OS model's responses are durable. Durability tracks survival where breadth inverts it.

In [ ]:
rs = onkos.response_vs_survival(ds, context=ctx, survival_link='survival_link.nsclc_os_growth_rate', t=t, n=400)
rows = [r for r in rs.rows if r['median_dor_weeks'] is not None]
top_orr = max(rows, key=lambda r: r['orr'])
longest_os = max(rows, key=lambda r: r['median_os_weeks'])
print(f"highest ORR : {top_orr['record_id'].split('.')[1]} (ORR {top_orr['orr']:.2f}, DoR {top_orr['median_dor_weeks']:.0f}, OS {top_orr['median_os_weeks']:.0f})")
print(f"longest OS  : {longest_os['record_id'].split('.')[1]} (ORR {longest_os['orr']:.2f}, DoR {longest_os['median_dor_weeks']:.0f}, OS {longest_os['median_os_weeks']:.0f})")
assert top_orr['median_os_weeks'] == min(r['median_os_weeks'] for r in rows)   # broadest = worst survivor
assert longest_os['median_dor_weeks'] > top_orr['median_dor_weeks']            # best survivor more durable

fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter([r['orr'] for r in rows], [r['median_dor_weeks'] for r in rows],
                c=[r['median_os_weeks'] for r in rows], cmap='viridis', s=140, edgecolor='black')
ax.set(xlabel='ORR (breadth)', ylabel='median DoR (durability, wk)', title='Breadth vs durability, coloured by OS')
fig.colorbar(sc, label='median OS under k_g (wk)')
plt.show()

## Guardrails

Population/trial-level only — no individual response duration. Censoring is surfaced (the observed median is a lower bound under heavy censoring), DoR carries the chain's propagated tier (out-of-context → D), and the breadth-vs-durability comparison ranks *models*, never treatments. Onkos uses DoR to explain the ORR-surrogate failure, not to claim DoR is a better surrogate.

In [ ]:
rr = objective_response_rate(ds, 'drug_effect.norton_simon.nsclc', context=ctx, t=t, n=300)
print(f"Norton-Simon: DoR {rr.median_dor_weeks:.0f} wk, {rr.dor_censored_fraction:.0%} censored (durable/eradicated)")
out = objective_response_rate(ds, 'resistance.claret_2009.tgi', context={'tumor_type':'melanoma','line':'first'}, t=t, n=80)
assert out.tier == 'D'
d = rr.to_dict()
assert 'median_dor_weeks' in d and d['NOT_FOR_CLINICAL_USE'] is True
print('guardrails OK | censoring surfaced, tier passthrough verified')